# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [3]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (16992, 22)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [4]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5162,16968810.29,3521.23,1169.14,26602
Audio,3949,8628316.89,2349.12,1072.44,15078
Mobilier,3936,7785225.19,2124.79,710.34,18274
Accessoire,3870,7196780.95,2009.71,831.52,14978


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [5]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie   montant  ca_categorie  part_dans_categorie
0  Informatique   457.881  1.696881e+07               0.0027
1      Mobilier   140.960  7.785225e+06               0.0018
2         Audio  2073.360  8.628317e+06               0.0240
3  Informatique  2281.000  1.696881e+07               0.0134
4      Mobilier  1904.490  7.785225e+06               0.0245

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [6]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,5738,P0027,0.165
telephone,2865,P0015,0.166
web,8389,P0038,0.170


### Partie autonome

In [ ]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.
# Q1. Chiffre d'affaires par catégorie et par canal


ca_categorie_canal = pd.pivot_table(
    sales,
    values='montant',
    index='categorie',
    columns='canal',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
    fill_value=0
)

display(ca_categorie_canal)


canal,boutique,telephone,web,Total
categorie,,,,
Accessoire,2.077403e+06,2.263645e+06,2.855733e+06,7.196781e+06
Audio,2.466857e+06,8.205815e+05,5.340878e+06,8.628317e+06
Informatique,6.115566e+06,4.752529e+06,6.100715e+06,1.696881e+07
Mobilier,3.285612e+06,2.424767e+06,2.074846e+06,7.785225e+06
Total,1.394544e+07,1.026152e+07,1.637217e+07,4.057913e+07


In [ ]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.

# Convertir la date en format datetime
sales['date_commande'] = pd.to_datetime(sales['date_commande'])
# Calculer les indicateurs pour chaque client
analyse_clients = sales.groupby('id_client').agg(
    nombre_commandes=('id_commande', 'nunique'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    categories_distinctes=('categorie', 'nunique')
).reset_index()

# Identifier les 10 meilleurs clients par chiffre d'affaires
top_10_clients = analyse_clients.sort_values(
    by='ca_total',
    ascending=False
).head(10)

# Afficher les résultats
display(analyse_clients)
display(top_10_clients)


,id_client,nombre_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,categories_distinctes
0,990000,6,8825.0360,1765.007200,2023-03-30 11:45:00,2024-11-25 00:00:00,4
1,990001,5,5462.4475,1092.489500,2023-06-16 00:00:00,2024-10-20 00:00:00,3
2,990002,7,7469.4635,1067.066214,2023-02-21 00:00:00,2024-10-16 00:00:00,4
3,990003,3,4725.2420,1575.080667,2024-02-20 00:00:00,2024-05-21 00:00:00,3
4,990004,4,6420.6105,2140.203500,2023-01-16 00:00:00,2024-11-18 00:00:00,4
...,...,...,...,...,...,...,...
3046,C02996,8,11280.8115,1611.544500,2023-06-24 00:00:00,2024-08-26 18:19:00,4
3047,C02997,6,1165.6940,291.423500,2023-04-06 00:00:00,2024-11-08 00:00:00,3
3048,C02998,7,6329.8655,904.266500,2023-03-31 00:00:00,2024-11-01 00:00:00,3
3049,C02999,8,9088.8895,1136.111187,2023-03-26 00:00:00,2024-12-15 00:00:00,4


,id_client,nombre_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,categories_distinctes
752,C00696,7,1.110047e+06,185007.767250,2023-03-13 00:00:00,2024-11-19 00:00:00,3
2379,C02327,4,9.078345e+05,226958.620125,2023-10-31 00:00:00,2024-12-28 14:36:00,2
336,C00279,10,8.769995e+05,87699.945400,2023-01-24 00:00:00,2024-11-05 00:00:00,4
2682,C02631,10,7.882669e+05,78826.687950,2023-01-12 00:00:00,2024-12-19 00:00:00,4
995,C00941,7,7.831793e+05,111882.758214,2023-01-29 00:00:00,2024-09-15 00:00:00,3
412,C00355,13,7.790788e+05,64923.237458,2023-04-09 16:29:00,2024-09-25 00:00:00,4
2153,C02101,6,7.507439e+05,125123.982667,2023-01-05 11:00:00,2024-02-03 00:00:00,3
860,C00806,5,7.072623e+05,176815.576500,2023-05-03 00:00:00,2024-08-26 00:00:00,2
2337,C02285,7,6.871645e+05,98166.358429,2023-01-10 10:43:00,2024-12-27 00:00:00,2
2088,C02036,3,6.418394e+05,213946.468000,2023-02-22 00:00:00,2023-07-02 00:00:00,2


In [21]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?


# Calculer la médiane du montant pour chaque catégorie
sales['mediane_categorie'] = sales.groupby('categorie')['montant'].transform('median')

# Identifier les commandes qui dépassent 150 % de la médiane
sales['depassement_50_pct'] = (
    sales['montant'] > sales['mediane_categorie'] * 1.5
)

# Calculer la part des commandes concernées
part_commandes = sales['depassement_50_pct'].mean() * 100

# Afficher les résultats
print("Nombre de commandes/lignes concernées :",
      sales['depassement_50_pct'].sum())

print("Part des commandes concernées :",
      round(part_commandes, 2), "%")

display(sales[['id_commande', 'categorie', 'montant',
               'mediane_categorie', 'depassement_50_pct']].head(10))


Nombre de commandes/lignes concernées : 5264
Part des commandes concernées : 30.98 %


,id_commande,categorie,montant,mediane_categorie,depassement_50_pct
0,CMD0003112,Informatique,457.8810,1169.140,False
1,CMD0000107,Mobilier,140.9600,710.336,False
2,CMD0010050,Audio,2073.3600,1072.440,True
3,CMD0001979,Informatique,2281.0000,1169.140,True
4,CMD0010513,Mobilier,1904.4900,710.336,True
5,CMD0015463,Audio,1419.0400,1072.440,False
6,CMD0001458,Mobilier,400.0525,710.336,False
7,CMD0005166,Informatique,1040.5870,1169.140,False
8,CMD0009796,Accessoire,2570.7000,831.521,True
9,CMD0010934,Audio,608.7800,1072.440,False


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [23]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 16992
Après jointure produits : 16992
Lignes sans produit correspondant : 75 (0.44%)
Identifiants fautifs : ['P9999']


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [24]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
16992 lignes -> 17287 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [25]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [ ]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).
# 1. Vérifier les nombres de lignes de départ
print("Nombre de lignes initial :")
print("sales     :", len(sales))
print("products  :", len(products))
print("customers :", len(customers))
print("stores    :", len(stores))

# 2. Fonction de fusion avec contrôle des clés orphelines
def fusionner_dimension(base, dimension, cle, nom_dimension):
    # Exclure les clés nulles de la table de référence
    dimension_clean = dimension.loc[dimension[cle].notna()].copy()

    # Vérifier que la clé de la dimension est unique
    if dimension_clean[cle].duplicated().any():
        raise ValueError(
            f"Clé {cle} non unique dans {nom_dimension}. "
            "Vérifier les doublons avant la fusion."
        )

    # Identifier les clés absentes de la dimension
    cles_base = base[cle]
    cles_reference = dimension_clean[cle]

    orphelines = cles_base.isna() | ~cles_base.isin(cles_reference)

    print(f"\n--- Fusion avec {nom_dimension} ---")
    print("Lignes avant :", len(base))
    print("Clés orphelines :", int(orphelines.sum()))
    print("Valeurs distinctes orphelines :",
          base.loc[orphelines, cle].nunique())

    # Éviter les conflits de noms : les colonnes communes sont suffixées
    resultat = base.merge(
        dimension_clean,
        on=cle,
        how='left',
        validate='many_to_one',
        indicator='_statut_fusion',
        suffixes=('', f'_{nom_dimension}')
    )

    print("Lignes après :", len(resultat))
    print("Lignes sans correspondance :",
          int((resultat['_statut_fusion'] == 'left_only').sum()))

    # Retirer la colonne technique de contrôle
    resultat = resultat.drop(columns='_statut_fusion')

    # Contrôle : une jointure many-to-one ne doit pas multiplier les ventes
    assert len(resultat) == len(base), (
        f"Le nombre de lignes a changé lors de la fusion avec {nom_dimension}."
    )

    return resultat


# 3. Préparer la table de départ
df_analytique = sales.copy()

# 4. Ajouter les informations sur les produits
df_analytique = fusionner_dimension(
    df_analytique, products, 'id_produit', 'products'
)

# 5. Ajouter les informations sur les clients
df_analytique = fusionner_dimension(
    df_analytique, customers, 'id_client', 'customers'
)

# 6. Ajouter les informations sur les magasins
df_analytique = fusionner_dimension(
    df_analytique, stores, 'id_magasin', 'stores'
)

# 7. Vérification finale
print("\n--- Résultat final ---")
print("Nombre de lignes dans sales :", len(sales))
print("Nombre de lignes analytiques :", len(df_analytique))
print("Nombre de colonnes :", len(df_analytique.columns))

display(df_analytique.head())


Nombre de lignes initial :
sales     : 16992
products  : 39
customers : 3060
stores    : 9

--- Fusion avec products ---
Lignes avant : 16992
Clés orphelines : 75
Valeurs distinctes orphelines : 1
Lignes après : 16992
Lignes sans correspondance : 75

--- Fusion avec customers ---
Lignes avant : 16992
Clés orphelines : 0
Valeurs distinctes orphelines : 0
Lignes après : 16992
Lignes sans correspondance : 0

--- Fusion avec stores ---
Lignes avant : 16992
Clés orphelines : 0
Valeurs distinctes orphelines : 0
Lignes après : 16992
Lignes sans correspondance : 0

--- Résultat final ---
Nombre de lignes dans sales : 16992
Nombre de lignes analytiques : 16992
Nombre de colonnes : 38


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison,montant,libelle,categorie,sous_categorie,cout_achat,marge_unitaire,taux_marge,montant_apres_remise,panier_moyen_client,commande_non_aboutie,ecart_prix_median_categorie,ca_categorie,part_dans_categorie,mediane_categorie,depassement_50_pct,libelle_products,categorie_products,sous_categorie_products,cout_achat_products,date_inscription,segment,ville,age,nom_magasin,ville_stores,region,surface_m2
0,CMD0003112,2024-01-18 19:53:00,C02300,P0009,M05,3,160.66,5.0,telephone,retourne,lille,457.881,Clavier modele 3,Informatique,Clavier,86.41,74.25,0.4622,457.881,1478.870333,1,-702.85,1.696881e+07,0.0027,1169.140,False,Clavier modele 3,Informatique,Clavier,86.41,2021-04-18,Particulier,PARIS,39.0,Boutique Bordeaux,Bordeaux,Nouvelle-Aquitaine,769
1,CMD0000107,2024-01-08 00:00:00,C02791,P0028,M03,2,70.48,0.0,boutique,livre,toulouse,140.960,Lampe modele 1,Mobilier,Lampe,34.81,35.67,0.5061,140.960,860.969571,0,-380.05,7.785225e+06,0.0018,710.336,False,Lampe modele 1,Mobilier,Lampe,34.81,2024-02-19,Particulier,PARIS,36.0,Boutique Marseille,Marseille,PACA,460
2,CMD0010050,2023-04-15 00:00:00,990031,P0017,M04,3,691.12,0.0,boutique,livre,marseille,2073.360,Enceinte modele 2,Audio,Enceinte,347.41,343.71,0.4973,2073.360,1170.664800,0,-37.84,8.628317e+06,0.0240,1072.440,True,Enceinte modele 2,Audio,Enceinte,347.41,2019-02-24,Professionnel,Bordeaux,NaN,Boutique Lille,Lille,Hauts-de-France,245
3,CMD0001979,2024-11-06 00:00:00,C01883,P0010,M04,4,570.25,0.0,web,retourne,marseille,2281.000,Souris modele 1,Informatique,Souris,283.64,286.61,0.5026,2281.000,1225.822375,1,-293.26,1.696881e+07,0.0134,1169.140,True,Souris modele 1,Informatique,Souris,283.64,2019-06-01,Particulier,Nantes,77.0,Boutique Lille,Lille,Hauts-de-France,245
4,CMD0010513,2024-11-02 00:00:00,C00571,P0024,M02,2,1058.05,10.0,web,livre,marseille,1904.490,Bureau modele 3,Mobilier,Bureau,554.59,503.46,0.4758,1904.490,1392.776667,0,607.52,7.785225e+06,0.0245,710.336,True,Bureau modele 3,Mobilier,Bureau,554.59,2021-01-02,Particulier,Strasbourg,44.0,Boutique Lyon,Lyon,Auvergne-Rhone-Alpes,853


In [28]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.


def safe_merge(left, right, **kwargs):
    # Récupérer le type de jointure et la clé
    how = kwargs.get('how', 'inner')
    on = kwargs.get('on')

    # Vérifier que les clés sont précisées
    if on is None:
        raise ValueError(
            "Précisez la clé de jointure avec le paramètre 'on'."
        )

    # Compter les lignes avant la jointure
    lignes_avant = len(left)

    # Effectuer la jointure avec un indicateur
    resultat = left.merge(
        right,
        **kwargs,
        indicator='_merge_status'
    )

    # Calculer le taux de clés non appariées côté gauche
    non_appariees = (
        resultat['_merge_status'] == 'left_only'
    ).sum()

    taux_non_appariees = (
        non_appariees / lignes_avant * 100
        if lignes_avant > 0 else 0
    )

    print(
        f"Taux de clés non appariées : "
        f"{taux_non_appariees:.2f} %"
    )

    # Vérifier que le nombre de lignes n'a pas changé
    if how == 'left' and len(resultat) != lignes_avant:
        raise ValueError(
            f"Le nombre de lignes a changé : "
            f"{lignes_avant} avant, {len(resultat)} après. "
            "Vérifiez les doublons dans les clés de droite."
        )

    # Supprimer la colonne technique
    resultat = resultat.drop(columns='_merge_status')

    return resultat


---
## Atelier 6.3 — Formats long et large (35 min)

In [ ]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{month:02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,342159.0,273765.0,304321.0,1201523.0,510234.0,318362.0,302848.0,1189096.0,459066.0,1002018.0,865577.0,427810.0
Audio,1152124.0,390711.0,352188.0,1055177.0,406065.0,1135499.0,545428.0,1044095.0,385388.0,394589.0,1132683.0,634370.0
Informatique,1785650.0,1534635.0,2676939.0,641790.0,2281109.0,709123.0,2890408.0,1025151.0,531817.0,553905.0,709323.0,1628961.0
Mobilier,831120.0,290465.0,349467.0,1111144.0,1320775.0,570811.0,341917.0,181351.0,1375521.0,291107.0,683401.0,438147.0


In [ ]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,342159.0
1,Audio,mois_01,1152124.0
2,Informatique,mois_01,1785650.0
3,Mobilier,mois_01,831120.0
4,Accessoire,mois_02,273765.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [37]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.



# Format long : CA par catégorie et par canal
long_ca = (
    sales.groupby(['categorie', 'canal'])['montant']
    .sum()
    .reset_index()
)

# Passage au format large avec pivot()
large_ca = long_ca.pivot(
    index='categorie',
    columns='canal',
    values='montant'
).fillna(0)

# Revenir au format long pour vérifier les valeurs
long_verification = (
    large_ca.rename_axis(columns='canal')
    .reset_index()
    .melt(
        id_vars='categorie',
        var_name='canal',
        value_name='montant'
    )
)

# Comparer les valeurs initiales et reconstituées
comparaison = long_ca.merge(
    long_verification,
    on=['categorie', 'canal'],
    how='outer',
    suffixes=('_initial', '_reconstitue')
)

comparaison['montant_reconstitue'] = (
    comparaison['montant_reconstitue'].fillna(0)
)

comparaison['identique'] = np.isclose(
    comparaison['montant_initial'],
    comparaison['montant_reconstitue']
)

print("Toutes les valeurs sont identiques :",
      comparaison['identique'].all())

display(large_ca)


Toutes les valeurs sont identiques : True


canal,boutique,telephone,web
categorie,,,
Accessoire,2.077403e+06,2263644.872,2.855733e+06
Audio,2.466857e+06,820581.500,5.340878e+06
Informatique,6.115566e+06,4752529.152,6.100715e+06
Mobilier,3.285612e+06,2424766.743,2.074846e+06


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [38]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [40]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.

# Créer le dossier de sortie
dossier = Path("data/processed/star")
dossier.mkdir(parents=True, exist_ok=True)

# 1. Construire les dimensions en supprimant les doublons
dim_customer = customers.drop_duplicates(subset=["id_client"]).copy()
dim_product = products.drop_duplicates(subset=["id_produit"]).copy()
dim_store = stores.drop_duplicates(subset=["id_magasin"]).copy()

# 2. Préparer la table de faits
fact_sales = sales.copy()

# Créer montant_net
if "montant_apres_remise" in fact_sales.columns:
    fact_sales["montant_net"] = fact_sales["montant_apres_remise"]
else:
    fact_sales["montant_net"] = (
        fact_sales["montant"] * (1 - fact_sales["remise_pct"] / 100)
    )

# 3. Récupérer cle_date depuis dim_date
print("Colonnes de dim_date :", dim_date.columns.tolist())

Colonnes de dim_date : ['date', 'cle_date', 'annee', 'mois', 'nom_mois', 'trimestre', 'semaine_iso', 'jour_semaine', 'nom_jour', 'est_weekend']


In [41]:
# Normaliser les dates pour la jointure
fact_sales["date_commande"] = pd.to_datetime(
    fact_sales["date_commande"]
).dt.normalize()

dim_date["date"] = pd.to_datetime(dim_date["date"]).dt.normalize()

# Ajouter la clé de date
fact_sales = fact_sales.merge(
    dim_date[["date", "cle_date"]],
    left_on="date_commande",
    right_on="date",
    how="left",
    validate="many_to_one"
)

# Sélectionner les colonnes demandées
fact_sales = fact_sales[
    [
        "cle_date",
        "id_client",
        "id_produit",
        "id_magasin",
        "quantite",
        "prix_unitaire",
        "remise_pct",
        "montant",
        "montant_net",
        "statut"
    ]
].copy()

display(fact_sales.head())

,cle_date,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,montant,montant_net,statut
0,20240118,C02300,P0009,M05,3,160.66,5.0,457.881,457.881,retourne
1,20240108,C02791,P0028,M03,2,70.48,0.0,140.960,140.960,livre
2,20230415,990031,P0017,M04,3,691.12,0.0,2073.360,2073.360,livre
3,20241106,C01883,P0010,M04,4,570.25,0.0,2281.000,2281.000,retourne
4,20241102,C00571,P0024,M02,2,1058.05,10.0,1904.490,1904.490,livre


In [42]:
def verifier_integrite(faits, dimension, cle, nom):
    cles_absentes = faits.loc[
        ~faits[cle].isin(dimension[cle]) & faits[cle].notna(),
        cle
    ].unique()

    cles_nulles = faits[cle].isna().sum()

    print(f"\n{nom}")
    print("Clés absentes de la dimension :", len(cles_absentes))
    print("Clés nulles dans la table de faits :", cles_nulles)

    return len(cles_absentes) == 0 and cles_nulles == 0


integrite_client = verifier_integrite(
    fact_sales, dim_customer, "id_client", "Clients"
)

integrite_produit = verifier_integrite(
    fact_sales, dim_product, "id_produit", "Produits"
)

integrite_magasin = verifier_integrite(
    fact_sales, dim_store, "id_magasin", "Magasins"
)

integrite_date = verifier_integrite(
    fact_sales, dim_date, "cle_date", "Dates"
)

print("\nIntégrité complète :",
      all([
          integrite_client,
          integrite_produit,
          integrite_magasin,
          integrite_date
      ]))


Clients
Clés absentes de la dimension : 0
Clés nulles dans la table de faits : 0

Produits
Clés absentes de la dimension : 1
Clés nulles dans la table de faits : 0

Magasins
Clés absentes de la dimension : 0
Clés nulles dans la table de faits : 0

Dates
Clés absentes de la dimension : 0
Clés nulles dans la table de faits : 0

Intégrité complète : False


In [43]:
fact_sales.to_parquet(dossier / "fact_sales.parquet", index=False)
dim_date.to_parquet(dossier / "dim_date.parquet", index=False)
dim_customer.to_parquet(dossier / "dim_customer.parquet", index=False)
dim_product.to_parquet(dossier / "dim_product.parquet", index=False)
dim_store.to_parquet(dossier / "dim_store.parquet", index=False)

print("Export terminé :", dossier.resolve())

Export terminé : C:\les-tp-data-manipulation\data\processed\star


In [39]:
from pathlib import Path

In [ ]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

def check_referential_integrity(fact_table, dimensions):
    """dimensions : dict {nom_colonne_cle: dataframe_dimension}"""
    ...  # TODO

---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.